# A multi-fuel combustion manifold

This example burns **three different fuels in three parallel branches** off a single air supply, then mixes the hot products back into one outlet.
It is built to stress the reacting mean-flow solver on a topology that is anything but a straight chain:

* one air stream is **split** three ways;
* each branch injects a **different fuel** (methane, propane, n-octane) at a **different equivalence ratio** (lean, near-stoichiometric, rich), and burns it at its own equilibrium flame;
* the branches reconverge through a small **web of junctions** — including a *branch-to-branch* link where one burnt stream feeds **two** downstream manifolds (a reconverging diamond), not just the final mixer;
* the geometry uses **area changes** (an isentropic diffuser, a sudden step) and a **loss** element, and the branch/duct areas are sized so the network sweeps a **wide range of Mach numbers** — from a lazy $M\approx0.04$ in the roomy branch up to $M\approx0.4$ at the tight hot outlet — while staying clear of $M=1$.

The whole thing is solved from a **bare `solve(prob)`** with no hand-built initial guess: the auto-seed propagates each feed stream through the graph and lands every edge at its adiabatic-mixing state.
That a network this irregular converges from the generic seed — and that each branch lands exactly on its standalone equilibrium flame temperature — is the robustness point.

> **One element set per network.** Every fuel here is a hydrocarbon, so the burnt species slate (`C, H, O, N`) is shared by every branch.
> Mixing fuels whose products span *different* elements across parallel branches (e.g. a carbon-free H₂ branch alongside the hydrocarbons) hits a deferred kernel path and is out of scope for this demo.

In [ ]:
import os, sys

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "fns")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go

from thermolib import ThermoInp, Thermo
from fns.composition import resolve_composition, enthalpy_mass
from fns.elements import catalog as cat
from fns.shell.network import Network
from fns.thermo.api import EQ_FROZEN, EQ_KERNEL
from fns.thermo.configure import equilibrium
from fns.derive import ES_MDOT, ES_P, ES_HT, ES_U, ES_T, ES_M

THERMO_INP = os.path.join(_root, "thermolib", "data", "thermo.inp")

## Species library

Air plus three hydrocarbon fuels, and the gas-phase products of their combustion.
Because every fuel contributes only carbon and hydrogen (air supplies oxygen and nitrogen), the element pool is `C, H, O, N` and every burnt branch carries the same species — no branch ends up with a zero-abundance element.

In [ ]:
AIR = {"O2": 0.21, "N2": 0.79}
FUELS = {
    "methane": {"CH4": 1.0},
    "propane": {"C3H8": 1.0},
    "n-octane": {"C8H18,n-octane": 1.0},
}

lib = ThermoInp(THERMO_INP).library(
    ["O2", "N2", "CH4", "C3H8", "C8H18,n-octane",
     "CO2", "H2O", "CO", "OH", "H", "O", "NO", "H2"]
)
gas = Thermo(lib)
print("elements:", lib.elements)
print("species :", [s.name for s in lib.species])

## The network

Node layout (element index in parentheses):

```
                                  diffuser   CH4        flame      bleed
  air ──► split ──┬──────────────► (2) ────► (3) ──────► (4) ─────► (11) ─┬─────────┐
   (0)     (1)    │ branch 1 (lean methane)                               │         │
                  │                                                       ▼         │
                  │                loss      C3H8       flame           mixA        │
                  ├──────────────► (5) ────► (6) ──────► (7) ─────────► (12) ──┐    │
                  │ branch 2 (~stoichiometric propane)                         ▼    │
                  │                                                        manifold │
                  │                step      C8H18      flame             (14) ─► out
                  └──────────────► (8) ────► (9) ──────► (10) ────────► (13) ──┘ (15)
                    branch 3 (rich n-octane)                             mixB ◄─────┘
```

The branch-to-branch link is the burnt **methane** stream at the *bleed splitter* `(11)`: it feeds **both** `mixA (12)` and `mixB (13)`, so the two sub-manifolds — and the final outlet — are a genuine reconverging diamond rather than three independent legs.

The thermo model is **frozen** up to and including each flame's inlet edge, and **equilibrium** on every edge downstream of a flame.
Areas are constant through each `source → flame` segment (those elements are compact / constant-area); all area changes live at the dedicated area-change nodes, the splitters, and the junctions.

In [ ]:
def build(m_air=2.0, m_ch4=0.043, m_c3h8=0.031, m_c8h18=0.024, Tair=600.0, Tfuel=300.0, p=4.0e5):
    """Assemble the multi-fuel manifold as a Network (areas fixed; flows are free)."""
    A_in = 0.012
    A1a, A1b = 0.010, 0.018    # branch 1 (CH4):  roomy  -> low Mach
    A2a, A2b = 0.006, 0.011    # branch 2 (C3H8): mid
    A3a, A3b = 0.0035, 0.0060  # branch 3 (C8H18): tight
    A_post, A_mix, A_out = 0.011, 0.012, 0.0098  # burnt legs / hot manifold / outlet

    els = [
        cat.mass_flow_inlet(m_air, Tair, composition=AIR, name="air"),                  # 0
        cat.splitter(name="split"),                                                     # 1
        cat.isentropic_area_change(name="diffuser"),                                    # 2
        cat.mass_source(m_ch4, Tfuel, composition=FUELS["methane"], name="CH4"),        # 3
        cat.equilibrium_flame(name="flame-CH4"),                                        # 4
        cat.loss(0.6, name="loss"),                                                     # 5
        cat.mass_source(m_c3h8, Tfuel, composition=FUELS["propane"], name="C3H8"),      # 6
        cat.equilibrium_flame(name="flame-C3H8"),                                       # 7
        cat.sudden_area_change(name="step"),                                            # 8
        cat.mass_source(m_c8h18, Tfuel, composition=FUELS["n-octane"], name="C8H18"),   # 9
        cat.equilibrium_flame(name="flame-C8H18"),                                      # 10
        cat.splitter(name="bleed"),                                                     # 11
        cat.junction(name="mixA"),                                                      # 12
        cat.junction(name="mixB"),                                                      # 13
        cat.junction(name="manifold"),                                                  # 14
        cat.pressure_outlet(p, Tt_backflow=Tair, composition=AIR, name="out"),          # 15
    ]
    edges = [
        (0, 1, A_in),
        (1, 2, A1a), (2, 3, A1b), (3, 4, A1b), (4, 11, A1b),    # branch 1
        (1, 5, A2a), (5, 6, A2b), (6, 7, A2b), (7, 12, A2b),    # branch 2
        (1, 8, A3a), (8, 9, A3b), (9, 10, A3b), (10, 13, A3b),  # branch 3
        (11, 12, A_post), (11, 13, A_post),                     # bleed -> both manifolds (diamond)
        (12, 14, A_mix), (13, 14, A_mix), (14, 15, A_out),      # recombine -> outlet
    ]
    frozen = {(0, 1), (1, 2), (2, 3), (3, 4), (1, 5), (5, 6), (6, 7), (1, 8), (8, 9), (9, 10)}
    edge_models = [EQ_FROZEN if (t, h) in frozen else EQ_KERNEL for (t, h, _a) in edges]

    Yair, _ = resolve_composition(lib, AIR)
    h_air = enthalpy_mass(lib, Yair, Tair)
    net = Network(
        gas=equilibrium(lib), p_ref=p, T_ref=Tair,
        mdot_ref=m_air, h_ref=max(abs(h_air), 1.0e3),
        nodes=els, edges=edges, edge_models=edge_models,
    )
    return net, edges

## Solve from the bare seed

No initial guess is supplied — `solve(prob)` builds its own from the feed streams discovered at assembly time.

In [ ]:
net, edges = build()
sol = net.solve()
prob = sol.problem
est = sol.table()

print("feed streams (transported mixture fractions):", list(prob.scalar_names))
print("converged:", sol.converged, " iterations:", sol.iterations)
print("Mach range: {:.3f} .. {:.3f}  (subsonic everywhere)".format(est[ES_M, :].min(), est[ES_M, :].max()))

## Per-edge state

Mass, temperature, pressure and Mach number on every edge, tagged frozen (unburnt) or burnt (equilibrium).

In [ ]:
edge_model = prob.edge_model
names = [f"{prob.node_names[t]}->{prob.node_names[h]}" for (t, h, _a) in edges]
tag = ["burnt" if m == EQ_KERNEL else "cold" for m in edge_model]

hdr = f"{'edge':<22}{'kind':>6}{'mdot':>9}{'T [K]':>9}{'p[bar]':>9}{'M':>8}{'u[m/s]':>9}"
print(hdr)
print("-" * len(hdr))
for e in range(prob.n_edges):
    print(
        f"{names[e]:<22}{tag[e]:>6}{est[ES_MDOT, e]:9.4f}{est[ES_T, e]:9.1f}"
        f"{est[ES_P, e] / 1e5:9.3f}{est[ES_M, e]:8.3f}{est[ES_U, e]:9.1f}"
    )

m_out = est[ES_MDOT, -1]
m_fuel = 0.043 + 0.031 + 0.024
print(f"\noutlet mdot = {m_out:.4f} kg/s   (air 2.000 + fuel {m_fuel:.3f} = {2.0 + m_fuel:.3f})")

## The network, drawn

The same diagram now comes straight from the shared topology plotter, `net.plot_topology`, with the solved state overlaid on the edges (the edges carry the state in FNS).
`color_by="T"` tints each edge by its temperature — cold reactant feeds read low, the burnt legs downstream of each flame read hot — and `width_by="mdot"` scales the arrow width by mass flow, so the fat air supply and the thin fuel injections are visible at a glance.
Hover any edge for its mass flow, temperature and Mach number.
The layout is automatic (longest-path layers, left to right); the bleed splitter feeding both `mixA` and `mixB` is the branch-to-branch diamond.

In [ ]:
net.plot_topology(
    solution=sol,
    color_by="T",
    width_by="mdot",
    title="Multi-fuel manifold — edge color = temperature, width ~ mass flow",
    width=920,
    height=560,
).show()

## Mach number across the network

The branch areas deliberately span a wide range of Mach number — and the solver keeps every edge comfortably subsonic.
The tight, hot outlet is the fastest station; the roomy methane branch is the slowest.

In [ ]:
order = np.argsort(est[ES_M, :])
fig = go.Figure(go.Bar(
    x=[names[e] for e in order],
    y=[est[ES_M, e] for e in order],
    marker_color=["#c0392b" if prob.edge_model[e] == EQ_KERNEL else "#2471a3" for e in order],
))
fig.add_hline(y=1.0, line=dict(color="black", dash="dash"),
              annotation_text="M = 1 (choke)", annotation_position="top left")
fig.update_layout(
    title="Edge Mach number (sorted) — well below choke",
    yaxis_title="Mach number", xaxis_tickangle=-60,
    width=900, height=460, margin=dict(l=40, r=20, t=50, b=140),
)
fig.show()

## Each branch lands on its own equilibrium flame temperature

The three branches run at different equivalence ratios, so they sit at three distinct flame temperatures.
For each one we take the air the solver routed into that branch, mix in its fuel, and compute the standalone constant-pressure equilibrium temperature (`equilibrate_HP`).
The network's burnt-edge temperature should match that reference — independent confirmation that the in-network flame is solving the same chemistry.

In [ ]:
def edge_index(edges, t, h):
    return next(i for i, (a, b, _a) in enumerate(edges) if a == t and b == h)


def hp_reference(air_mdot, fuel_spec, fuel_mdot, Tair=600.0, Tfuel=300.0, p=4.0e5):
    """Standalone constant-pressure equilibrium T of (branch air + its fuel)."""
    Yair, _ = resolve_composition(lib, AIR)
    Yf, _ = resolve_composition(lib, fuel_spec)
    m = air_mdot + fuel_mdot
    Ymix = (air_mdot * Yair + fuel_mdot * Yf) / m
    h_mix = (air_mdot * enthalpy_mass(lib, Yair, Tair) + fuel_mdot * enthalpy_mass(lib, Yf, Tfuel)) / m
    Z = gas.elemental_mass_fractions(Ymix)
    return gas.equilibrate_HP(Z, h_mix, p, T_guess=2000.0).T


branches = [
    ("methane (lean)", "methane", 0.043, (1, 2), (4, 11)),
    ("propane (~stoich)", "propane", 0.031, (1, 5), (7, 12)),
    ("n-octane (rich)", "n-octane", 0.024, (1, 8), (10, 13)),
]

print(f"{'branch':<20}{'air [kg/s]':>11}{'phi':>7}{'T_network':>11}{'T_HP ref':>10}{'diff':>8}")
print("-" * 67)
# stoichiometric mass air/fuel for phi
AFR_ST = {"methane": 17.16, "propane": 15.60, "n-octane": 15.03}
for label, fuel, mf, head, burnt in branches:
    m_air_branch = est[ES_MDOT, edge_index(edges, *head)]
    T_net = est[ES_T, edge_index(edges, *burnt)]
    T_ref = hp_reference(m_air_branch, FUELS[fuel], mf)
    phi = AFR_ST[fuel] / (m_air_branch / mf)
    print(f"{label:<20}{m_air_branch:>11.4f}{phi:>7.2f}{T_net:>11.1f}{T_ref:>10.1f}{T_net - T_ref:>8.2f}")

## Conservation across the whole manifold

Two global checks that must hold regardless of topology:

* **mass** — outlet flow equals air plus all injected fuel;
* **total enthalpy** — the outlet enthalpy flux equals the sum of the inflow enthalpy fluxes (the flames are adiabatic; heat release is *internal* to the absolute-enthalpy datum, so the formation-inclusive total enthalpy is conserved end to end).

In [ ]:
Yair, _ = resolve_composition(lib, AIR)
h_air_in = enthalpy_mass(lib, Yair, 600.0)
m_air = 2.0
H_in = m_air * h_air_in
m_fuel_total = 0.0
for _label, fuel, mf, _head, _burnt in branches:
    Yf, _ = resolve_composition(lib, FUELS[fuel])
    H_in += mf * enthalpy_mass(lib, Yf, 300.0)
    m_fuel_total += mf

H_out = est[ES_MDOT, -1] * est[ES_HT, -1]
print(f"mass  in/out : {m_air + m_fuel_total:.6f} / {est[ES_MDOT, -1]:.6f} kg/s"
      f"   (rel err {abs(est[ES_MDOT, -1] - m_air - m_fuel_total) / (m_air + m_fuel_total):.2e})")
print(f"enthalpy flux in/out : {H_in:.1f} / {H_out:.1f} W"
      f"   (rel err {abs(H_out - H_in) / abs(H_in):.2e})")

## Robustness: sweep the air supply

The same network, re-solved from the bare seed across a range of total air flow.
The split, the per-branch combustion and the recombination all re-settle on their own; the outlet stays subsonic throughout, and the maximum Mach in the network rises smoothly as the manifold is pushed harder.

In [ ]:
airs = np.linspace(1.4, 2.6, 13)
M_out, M_max, T_out, iters = [], [], [], []
for ma in airs:
    sol_i = build(m_air=ma)[0].solve()
    st = sol_i.table()
    M_out.append(st[ES_M, -1])
    M_max.append(st[ES_M, :].max())
    T_out.append(st[ES_T, -1])
    iters.append(sol_i.iterations if sol_i.converged else np.nan)

print("all converged:", all(np.isfinite(iters)), " iteration count range:",
      int(np.nanmin(iters)), "..", int(np.nanmax(iters)))

fig = go.Figure()
fig.add_trace(go.Scatter(x=airs, y=M_max, mode="lines+markers", name="max Mach in network"))
fig.add_trace(go.Scatter(x=airs, y=M_out, mode="lines+markers", name="outlet Mach"))
fig.add_hline(y=1.0, line=dict(color="black", dash="dash"),
              annotation_text="M = 1", annotation_position="top left")
fig.update_layout(
    title="Air-supply sweep — network stays subsonic",
    xaxis_title="air supply [kg/s]", yaxis_title="Mach number",
    width=820, height=460, margin=dict(l=50, r=20, t=50, b=50),
    legend=dict(x=0.01, y=0.99),
)
fig.show()

## Takeaways

* A genuinely irregular reacting network — one air feed, three fuels, three equivalence ratios, area changes, a loss, and a reconverging branch-to-branch diamond — solves from a **single generic seed** with no per-case tuning.
* Each branch independently reproduces its **standalone equilibrium flame temperature**, and mass and total enthalpy are conserved across the whole manifold to round-off.
* The geometry sweeps Mach from $\sim0.04$ to $\sim0.4$ and the solver holds the flow **subsonic** across an air-supply sweep — the flexibility and robustness the reacting mean-flow layer is built for.